# Oil & gas production decline curve

Pull the full monthly production history for a long-lived well and plot
oil, gas, and water volumes through time. This is the canonical "decline
curve" landmen and mineral owners look at to understand lease
performance and estimate remaining reserves.

Uses `sondio.oilgas.production(well_id)`, which resolves at well-level
where the source reports per-well volumes, or falls back to the
underlying lease where only lease-level data is published.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1.2,<0.2' matplotlib

import sondio
# sondio >= 0.1.2 reads your key from Colab Secrets (🔑 sidebar),
# Kaggle Secrets, SONDIO_API_KEY env var, or ~/.sondio/config — in that
# order. Set whichever fits your environment.
print(f"sondio {sondio.__version__}")

In [ ]:
# Anchor well: a Kansas oil well (BEREXCO, Butler County) with a long
# monthly history. Swap `WELL_ID` for any `external_id` or Sondio UUID —
# the API accepts either. Colorado, New Mexico, North Dakota, and
# Pennsylvania have the deepest per-well coverage.
WELL_ID = "15-083-21176"  # BEREXCO LLC — Labette County, KS
well_row = sondio.oilgas.wells(search=WELL_ID, limit=1).iloc[0]
print(f"{well_row['well_name']} — {well_row['operator_name']} ({well_row['external_id']})")
well_row[["well_name", "operator_name", "well_type", "well_status", "locality", "subdivision"]]

In [ ]:
# Pull the monthly production history. Default is 24 months; ask for 180
# (15 years) to see the full decline curve.
prod = sondio.oilgas.production(WELL_ID, months=180)
prod = prod.sort_values("production_date").reset_index(drop=True)
print(f"{len(prod)} months of production, earliest {prod['production_date'].min().date()}")
prod.head()

In [ ]:
# Three-panel decline curve — oil, gas, water on the same x axis.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, col, color, label in zip(
    axes,
    ["oil_bbl", "gas_mcf", "water_bbl"],
    ["#1a6e3d", "#d68b2a", "#2a6cad"],
    ["Oil (bbl/mo)", "Gas (mcf/mo)", "Water (bbl/mo)"],
):
    ax.fill_between(prod["production_date"], 0, prod[col].fillna(0), color=color, alpha=0.6)
    ax.plot(prod["production_date"], prod[col].fillna(0), color=color, linewidth=0.8)
    ax.set_ylabel(label); ax.grid(alpha=0.2)
axes[0].set_title(f"{well_row['well_name']} — {well_row['operator_name']} ({well_row['external_id']})")
axes[-1].set_xlabel("production month")
plt.tight_layout(); plt.show()

In [ ]:
# Boe-equivalent lifetime total and peak month.
prod["boe"] = prod["oil_bbl"].fillna(0) + prod["gas_mcf"].fillna(0) / 6.0
peak = prod.loc[prod["boe"].idxmax()]
print(f"Cumulative BOE: {prod['boe'].sum():,.0f}")
print(f"Peak month: {peak['production_date'].date()} — {peak['boe']:,.0f} BOE")

## Next steps
- Loop over top-N wells for a single operator and plot combined monthly BOE.
- Pass `as_of=` (Pro+ tier) to replay production as it would have looked on a historical date — useful for back-testing acquisition models.
- Cross with `sondio.us.phmsa.pipeline_incidents(...)` to flag wells feeding into pipelines with recent incidents.
- Related: [earthquake-well-proximity](../cross-vertical/earthquake-well-proximity.ipynb).